In [1]:
from mobgap import gait_sequences

from functions import *

import numpy as np
import os
from scipy.signal import butter, filtfilt, find_peaks
from scipy.linalg import logm
from scipy.io import savemat, loadmat
import scipy.io as spio
import pandas as pd
from datetime import datetime
from vedo import Points, Plotter, Line, Grid
from IPython.display import Video
import imageio.v2 as imageio
import shutil
import seaborn as sns
import stumpy
import itertools

import matplotlib
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error

from mobgap.data import GenericMobilisedDataset
from mobgap.pipeline import MobilisedPipelineImpaired
from mobgap.aggregation import get_mobilised_dmo_thresholds
from mobgap.gait_sequences import GsdAdaptiveIonescu, GsdIluz, GsdIonescu
from mobgap.initial_contacts import IcdHKLeeImproved, IcdIonescu, IcdShinImproved

import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning) 
warnings.filterwarnings("ignore", category=UserWarning)

C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\.venv\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator SVC from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please re

In [ ]:
# should videos be generated?
generate_videos = False
prints = False
regenerate_data_files = False

base_path = r"C:\Users\ac4jmi\Desktop\DMO4LNC\dmo4lnc-analysis\Dataset"

# Loop through all the participant folders syncing axivity sensors and mocap timings
for cohort in os.listdir(base_path):
    cohort_path = os.path.join(base_path, cohort)
    if os.path.isdir(cohort_path):
        for participant in os.listdir(cohort_path):
            participant_path = os.path.join(cohort_path, participant)
            if os.path.isdir(participant_path):
                participant_path = os.path.join(participant_path, 'Lab\\')
                print(participant_path)  # full path to each session folder
                if (not os.path.exists(os.path.join(participant_path, 'Laboratory\\data.mat'))) or (regenerate_data_files):
                    generate_trimmed_resampled_axivity(participant_path, prints = prints)
                    align_imu_mocap(participant_path, prints = prints)
                    if generate_videos:
                        video_path = os.path.join(participant_path, 'Motion Capture Data')
                        video_path = os.path.join(video_path, '')
                        tsv_files = [
                            os.path.splitext(f)[0]
                            for f in os.listdir(video_path)
                            if f.endswith('.tsv') and not f.startswith('synced')
                        ]
                        
                        for file_name in tsv_files:
                            generate_video_from_raw_data(file_name, video_path, prints = False)
                            Video(video_path+file_name+".mp4", embed=True, width=608)
                else:
                    print("data.mat already exists for this file!")

print("Preprocessing Done!")

# Now, run the MATLAB script to obtain standards...

### Then continue:

# Calculating DMOs from MobGap and Motion Capture

In [2]:
def assess_pipeline(dataset, pipeline, cohorts, subjects_to_ignore, all_tests, prints = False, plots = False):

    all_metrics = pd.DataFrame(columns=["cohort", "subject", "file_name", "wb_ID", "wb_TPs", "wb_FPs", "wb_FNs", "wb_TNs", "mobgap_ICs", "mocap_ICs",
                                    "IC_TPs", "IC_FPs", "IC_FNs", "stride_length_mobgap", "stride_length_mocap", "cadence_mocap", "cadence_mobgap"])

    # figure size settings
    fig_scaler = 10
    matplotlib.rcParams.update({'font.size': 2.5*fig_scaler})

    for cohort in cohorts:
        data = dataset.get_subset(cohort=cohort)
        subjects = list({row[1] for row in dataset.group_labels if row[0] == cohort})
        subjects = list(set(subjects) - set(subjects_to_ignore)) # remove problem subjects
        # overwrite by uncommenting:
        #subjects = ['234']
        if prints:
            print(f"Subjects for this cohort: {subjects}")

        for subject in subjects:
            start_location = os.path.join(os.getcwd(), "Dataset")
            dat_files = get_paths_with_extension(extension="data.mat", start_location=start_location, folders_to_ignore=["Home"])

            # convert mat file to a dictionary
            print(f"Subject {subject}")
            if prints:
                print("Reading and converting mat file...")
                print()
            index = next((i for i, path in enumerate(dat_files) if f"\\{subject}\\" in path), None) # get the index of the chosen subject
            mat = loadmat_fixed(dat_files[index])


            for test_to_compare in all_tests:
                # run pipeline on current test for current subject

                # init variables for monitoring performance of mobgap
                cur_sub_metrics = pd.Series(index=["cohort", "subject", "file_name", "wb_ID", "wb_TPs", "wb_FPs", "wb_FNs", "wb_TNs", "mobgap_ICs", "mocap_ICs",
                                                   "IC_TPs", "IC_FPs", "IC_FNs", "stride_length_mobgap", "stride_length_mocap", "cadence_mocap", "cadence_mobgap"], dtype = 'object')

                test = data.get_subset(Test=test_to_compare, subject_id = str(subject))[0]
                pipeline = pipeline.safe_run(test) # pipeline now stores all the results for all wbs

                # mat file shortcuts to reduce variable sizes
                cur_mat_root = mat["data"]["TimeMeasure1"][test_to_compare]["Trial1"]
                cur_file_name = cur_mat_root["FileName"]

                # wb sizes
                mobgap_all_wbs = len(pipeline.raw_ic_list_['ic'].index.get_level_values(0))
                if not mobgap_all_wbs == 0:
                    wbs_in_current_test_mobgap = (max(pipeline.raw_ic_list_['ic'].index.get_level_values(0))+1)
                else:
                    wbs_in_current_test_mobgap = 0

                if "Stereophoto" in cur_mat_root["Standards"]:
                    wbs_in_current_test_mocap = len(cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"])
                    if wbs_in_current_test_mocap >= 60: # magic number caused by matlab implementation. outside scope to delve into the matlab code
                        wbs_in_current_test_mocap = 1
                else:
                    wbs_in_current_test_mocap = 0

                if prints:
                    print()
                    print("######################### " + cur_file_name + " #########################")

                # add the stuff for this test to the metrics series
                cur_sub_metrics["cohort"] = cohort
                cur_sub_metrics["subject"] = subject
                cur_sub_metrics["file_name"] = cur_file_name
                cur_sub_metrics[["wb_TPs", "wb_FPs", "wb_FNs", "wb_TNs", "mobgap_ICs", "mocap_ICs", "IC_TPs", "IC_FPs", "IC_FNs"]] = 0


                if test_to_compare in ["Test1"]:
                    # if there are walking bouts detected in standing, we have false positives, else true negatives
                    if wbs_in_current_test_mobgap >= 1:
                        cur_sub_metrics["wb_FPs"] += wbs_in_current_test_mobgap
                    else:
                        cur_sub_metrics["wb_TNs"] += 1

                    cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                    all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)

                else:
                    # for all other activities, process each wb
                    mocap_timestamp = cur_mat_root["SU"]["LowerBack"]["Timestamp"]
                    if (wbs_in_current_test_mocap == 1) and (wbs_in_current_test_mobgap == 1):
                        cur_sub_metrics["wb_ID"] = 1
                        cur_sub_metrics["wb_TPs"] = 1
                        # get the indices of all IC events detected by mobgap and mocap
                        mocap_ic_events = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["InitialContact_Event"]
                        mocap_event_indices = [get_closest_index(mocap_timestamp, ic_event) for ic_event in mocap_ic_events]
                        mobgap_event_indices = pipeline.raw_ic_list_['ic'].values

                        # reduce the IC events to only true positives and plot them
                        reduced_mobgap_event_indices, mask, mobgap_mask, mocap_mask, metrics = match_closest_unique(mocap_event_indices, mobgap_event_indices, prints = prints)
                        #plot_subject_wbs(test, mocap_event_indices, mobgap_event_indices)

                        # store the IC metrics
                        cur_sub_metrics["mobgap_ICs"] = len(mobgap_event_indices)
                        cur_sub_metrics["mocap_ICs"] = len(mocap_event_indices)
                        for k, v in metrics.items():
                            cur_sub_metrics[k] = v

                        # get the DMOs for the current trial
                        cur_sub_metrics["stride_length_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["AverageStrideLength"]
                        cur_sub_metrics["stride_length_mobgap"] = pipeline.per_wb_parameters_["stride_length_m"].values
                        cur_sub_metrics["cadence_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["AverageCadence"]
                        cur_sub_metrics["cadence_mobgap"] = pipeline.per_wb_parameters_["cadence_spm"].values
                        cur_sub_metrics["walking_speed_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["AverageSpeed"]
                        cur_sub_metrics["walking_speed_mobgap"] = pipeline.per_wb_parameters_["walking_speed_mps"].values

                        cur_sub_metrics["wb_start_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["Start"]
                        cur_sub_metrics["wb_start_mobgap"] = pipeline.per_wb_parameters_["start"].values/100
                        cur_sub_metrics["wb_end_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["End"]
                        cur_sub_metrics["wb_end_mobgap"] = pipeline.per_wb_parameters_["end"].values/100
                        cur_sub_metrics["wb_duration_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"]["Duration"]
                        cur_sub_metrics["wb_duration_mobgap"] = pipeline.per_wb_parameters_["duration_s"].values


                        cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                        all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)

                    else:
                        # more than 1 CWB means we need to loop through them and treat them like seperate tests
                        # first, check if mobgap and mocap both detected the same number of walking bouts
                        if wbs_in_current_test_mocap == wbs_in_current_test_mobgap:
                            if prints:
                                print(f"{wbs_in_current_test_mocap} mocap wbs detected!")

                            for wb in range(wbs_in_current_test_mocap):
                                if prints:
                                    print(f"WB: {wb}")
                                cur_sub_metrics["wb_ID"] = wb+1 # 0 indexed
                                cur_sub_metrics["wb_TPs"] = 1

                                # get the indices of all IC events detected by mobgap and mocap for this walking bout
                                mocap_ic_events = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["InitialContact_Event"]
                                mocap_event_indices = [get_closest_index(mocap_timestamp, ic_event) for ic_event in mocap_ic_events]
                                mobgap_event_indices = pipeline.raw_ic_list_['ic'].loc[wb].values

                                # reduce the IC events to only true positives and plot them
                                reduced_mobgap_event_indices, mask, mobgap_mask, mocap_mask, metrics = match_closest_unique(mocap_event_indices, mobgap_event_indices)
                                if plots:
                                    plot_subject_wbs(test, mocap_event_indices, mobgap_event_indices)

                                # store the IC metrics
                                cur_sub_metrics["mobgap_ICs"] = len(mobgap_event_indices)
                                cur_sub_metrics["mocap_ICs"] = len(mocap_event_indices)
                                for k, v in metrics.items():
                                    cur_sub_metrics[k] = v

                                # get the DMOs for the current wb
                                # helper function to safely extract mobgap parameters regardless of index type or length
                                def get_mobgap_param(df, col, wb_idx):
                                    if df is None or df.empty or col not in df.columns:
                                        return np.nan
                                    if wb_idx in df.index:
                                        val = df.loc[wb_idx, col]
                                    elif isinstance(wb_idx, int) and wb_idx < len(df):
                                        val = df[col].iloc[wb_idx]
                                    else:
                                        return np.nan
                                    return val.item() if hasattr(val, "item") and not hasattr(val, "__len__") else val

                                cur_sub_metrics["stride_length_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["AverageStrideLength"]
                                cur_sub_metrics["cadence_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["AverageCadence"]
                                cur_sub_metrics["walking_speed_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["AverageSpeed"]
                                if not cur_sub_metrics["walking_speed_mocap"]:
                                    cur_sub_metrics["walking_speed_mocap"] = np.nan

                                cur_sub_metrics["stride_length_mobgap"] = get_mobgap_param(pipeline.per_wb_parameters_, "stride_length_m", wb)
                                cur_sub_metrics["cadence_mobgap"] = get_mobgap_param(pipeline.per_wb_parameters_, "cadence_spm", wb)
                                cur_sub_metrics["walking_speed_mobgap"] = get_mobgap_param(pipeline.per_wb_parameters_, "walking_speed_mps", wb)

                                # get the start, end, and duration of each walking bout
                                cur_sub_metrics["wb_start_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["Start"]
                                cur_sub_metrics["wb_start_mobgap"] = get_mobgap_param(pipeline.per_wb_parameters_, "start", wb)/100 # divide by 100 to match mocap
                                cur_sub_metrics["wb_end_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["End"]
                                cur_sub_metrics["wb_end_mobgap"] = get_mobgap_param(pipeline.per_wb_parameters_, "end", wb)/100 # divide by 100 to match mocap
                                cur_sub_metrics["wb_duration_mocap"] = cur_mat_root["Standards"]["Stereophoto"]["ContinuousWalkingPeriod"][wb]["Duration"]
                                cur_sub_metrics["wb_duration_mobgap"] = get_mobgap_param(pipeline.per_wb_parameters_, "duration_s", wb)

                                cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                                all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)

                        else:
                            if prints:
                                print(f"{wbs_in_current_test_mocap=}")
                                print(f"{wbs_in_current_test_mobgap=}")
                            # mismatched wbs - report false positive/false negatives
                            if wbs_in_current_test_mocap > wbs_in_current_test_mobgap:
                                cur_sub_metrics["wb_FNs"] = (wbs_in_current_test_mocap - wbs_in_current_test_mobgap)
                            else:
                                cur_sub_metrics["wb_FPs"] = (wbs_in_current_test_mobgap - wbs_in_current_test_mocap)

                            cur_sub_metrics = cur_sub_metrics.infer_objects() # automatically get dtypes
                            all_metrics = pd.concat([all_metrics, cur_sub_metrics.to_frame().T], ignore_index=True)

    # convert NAN columns to floats
    cols_with_nan = [c for c in all_metrics.columns if all_metrics[c].isna().any()]

    # convert lists to single values
    all_metrics[cols_with_nan] = all_metrics[cols_with_nan].map(
        lambda x: x[0] if isinstance(x, list) and len(x) > 0 else (np.nan if isinstance(x, list) else x)
    )

    all_metrics[cols_with_nan] = all_metrics[cols_with_nan].astype(float)

    # format to 1dp and print
    pd.options.display.float_format = lambda x: f"{x:.1f}"

    return all_metrics

In [3]:
# get mobgap dataset
paths_list = get_paths_with_extension("data.mat",
                                      start_location=os.path.join(os.getcwd(), "Dataset"),
                                      folders_to_ignore=["Home"])

dataset = get_mobilised_dataset(paths_list, parent_folders_as_metadata=["cohort", "subject_id", "location", "sensor"])
print(dataset)

cohorts = ["CP", "PSP"] # can add "HA"

subjects_to_ignore = ['969', '921'] # 969 was heavily gait impaired, 921 was not actually cp

all_tests = ["Test" + str(i) for i in range(1, 10)]

# get the HA healthy thresholds and rename them to "CP" so that we can use them to do some basic thresholding on the data
ha_thresholds = get_mobilised_dmo_thresholds().xs("HA", level=1, drop_level=False)
new_index = pd.MultiIndex.from_tuples([(dmo, cohort) for dmo, _ in ha_thresholds.index for cohort in cohorts], names=ha_thresholds.index.names)
duplicated_values = pd.concat([ha_thresholds] * len(cohorts), axis=0).reset_index(drop=True)
multi_cohort_thresholds = pd.DataFrame(duplicated_values.values, index=new_index, columns=ha_thresholds.columns)

#default_pipeline = MobilisedPipelineImpaired(dmo_thresholds=multi_cohort_thresholds) # thresholding set to HA equivalent

default_pipeline = MobilisedPipelineImpaired(
    gait_sequence_detection=GsdIluz(window_length_s = 4.899691636927754,
                                    window_overlap = 0.5808661183690179,
                                    std_activity_threshold = -2.445024251244571,
                                    mean_activity_threshold = -0.44382975752720877,
                                    acc_v_standing_threshold = -7.6531159461273495,
                                    sin_template_freq_hz = 1.0533455063274262,
                                    allowed_acc_v_change_per_window = 0.715261294298543,
                                    min_gsd_duration_s = 4.55958724725937,
                                    use_original_peak_detection = True),
    dmo_thresholds=multi_cohort_thresholds,
)

all_metrics = assess_pipeline(dataset, default_pipeline, cohorts, subjects_to_ignore, all_tests, prints = False)

display(all_metrics)
# save all metrics as a csv so we don't have to rerun this all the time
all_metrics.to_csv("In-Lab Parameters\\default_pipeline_output.csv")


Building dataset from .mat files...
GenericMobilisedDataset [122 groups/rows]

        cohort subject_id location      sensor   TimeMeasure   Test   Trial
   0        CP        234      Lab  Laboratory  TimeMeasure1  Test1  Trial1
   1        CP        234      Lab  Laboratory  TimeMeasure1  Test2  Trial1
   2        CP        234      Lab  Laboratory  TimeMeasure1  Test3  Trial1
   3        CP        234      Lab  Laboratory  TimeMeasure1  Test4  Trial1
   4        CP        234      Lab  Laboratory  TimeMeasure1  Test5  Trial1
   ..      ...        ...      ...         ...           ...    ...     ...
   117  Stroke        921      Lab  Laboratory  TimeMeasure1  Test5  Trial1
   118  Stroke        921      Lab  Laboratory  TimeMeasure1  Test6  Trial1
   119  Stroke        921      Lab  Laboratory  TimeMeasure1  Test7  Trial1
   120  Stroke        921      Lab  Laboratory  TimeMeasure1  Test8  Trial1
   121  Stroke        921      Lab  Laboratory  TimeMeasure1  Test9  Trial1
   
   [1

ValueError: setting an array element with a sequence.